# 3편 해외 손님에게 가격을 알려 주자
**쉽게 따라 해 보면서 배우는 파이썬**

위에서부터 한 셀씩 `Shift + Enter`. 설명은 사이트의 3편 페이지에 있다.

## 3-1 글자 가격을 숫자로

In [ ]:
# 3편 실습 파일 묶음(zip)을 올리고 푼다
from google.colab import files
import zipfile, os
up = files.upload()
zipfile.ZipFile(list(up)[0]).extractall()
print(sorted(f for f in os.listdir() if f.startswith(("shop_", "fx_", "ai_"))))

In [ ]:
import pandas as pd
enc = "utf-8-sig"
items = pd.read_csv("shop_items.csv", encoding=enc)
items["krw"] = pd.to_numeric(items["price"].str.replace("₩", "").str.replace(",", ""), errors="coerce")
print(items)
print("no price:", items.loc[items["krw"].isna(), "item"].tolist())
items = items.dropna(subset=["krw"])      # 가격이 없는 상품은 가격표에서 뺀다

## 3-2 yfinance 로 환율 받기

In [ ]:
# Colab 에 없으면 설치: !pip install yfinance
import yfinance as yf
tickers = ["USDKRW=X", "JPYKRW=X", "CNYKRW=X"]      # 1달러 · 1엔 · 1위안 = 몇 원
live = yf.download(tickers, start="2025-01-02", end="2026-01-01", progress=False)
print("rows:", len(live), "/ empty:", live.empty)
if live.empty:
    print("받지 못했다 → 백업 CSV 를 쓴다")
else:
    print(live["Close"].tail(3))

In [ ]:
fx = pd.read_csv("fx_2025_backup.csv", encoding=enc, parse_dates=["Date"], index_col="Date")
print(fx.shape)
print(fx.head(3))
print(fx.tail(2))

## 3-3 통화 세 개를 한 번에

In [ ]:
# 가로 표 → 세로 표: 통화 이름이 한 열(ticker)로 내려온다
long = fx.reset_index().melt(id_vars="Date", var_name="ticker", value_name="krw")
print(long.head(4))
long["month"] = long["Date"].dt.month
monthly = long.groupby(["month", "ticker"])["krw"].mean().unstack("ticker").round(2)
print(monthly)

## 3-4 곱할까 나눌까, 그리고 100엔

In [ ]:
rate = fx.iloc[-1]                     # 마지막 영업일 환율
print(rate)
# 1달러 = 1,4xx원 이므로, 원 → 달러는 '나누기'
tag = items[["item", "krw"]].copy()
tag["usd"] = (tag["krw"] / rate["USDKRW=X"]).round(2)
tag["jpy"] = (tag["krw"] / rate["JPYKRW=X"]).round(0)      # yfinance 는 1엔 기준
tag["cny"] = (tag["krw"] / rate["CNYKRW=X"]).round(1)
print(tag)
print("bank notice style: 100 JPY =", round(rate["JPYKRW=X"] * 100, 1), "KRW")

## 3-5 주말엔 환율이 없다 · CH-10

In [ ]:
orders = pd.read_csv("shop_orders_global.csv", encoding=enc, parse_dates=["date"])
orders["fc"] = orders["qty"] * orders["unit_price"]          # 그 나라 돈으로 낸 금액
print(orders.head(3))
# 영업일 환율만 그대로 붙이면?
t1 = orders.merge(fx, left_on="date", right_index=True, how="left")
print("orders:", len(orders), "/ no rate:", t1["USDKRW=X"].isna().sum())
print(t1.loc[t1["USDKRW=X"].isna(), "date"].dt.day_name().value_counts())

In [ ]:
cal = pd.date_range("2025-01-01", "2025-12-31")
fxd = fx.reindex(cal).ffill().bfill()
fxd.index.name = "date"
print(fxd.loc["2025-01-03":"2025-01-06", ["USDKRW=X"]])
t2 = orders.merge(fxd, left_on="date", right_index=True, how="left")
print("no rate now:", t2["USDKRW=X"].isna().sum())

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(fxd.index, fxd["USDKRW=X"], color="#D98E22", linewidth=1, label="Calendar (weekend filled)")
ax.plot(fx.index, fx["USDKRW=X"], ".", color="#1F3A5F", markersize=3, label="Business days")
ax.set(title="CH-10 USD/KRW 2025 (practice backup)", ylabel="KRW per USD")
ax.legend()
plt.show()

## 3-6 단순 평균 환율 vs 주문 가중 평균 환율 · CH-11

In [ ]:
us = t2[t2["currency"] == "USD"].copy()
us["krw"] = us["fc"] * us["USDKRW=X"]                  # 그날 환율로 바꾼 원화
us["month"] = us["date"].dt.month
g = us.groupby("month")[["fc", "krw"]].sum()
g["weighted"] = (g["krw"] / g["fc"]).round(1)                 # 주문 금액 가중 평균 환율
g["simple"] = fxd.groupby(fxd.index.month)["USDKRW=X"].mean().round(1)   # 달력 평균
g["diff"] = (g["weighted"] - g["simple"]).round(1)
print(g[["fc", "simple", "weighted", "diff"]])
print("year: simple", round(fxd["USDKRW=X"].mean(), 1), "/ weighted", round(us["krw"].sum() / us["fc"].sum(), 1))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(g.index, g["diff"], color=["#C9382A" if v > 0 else "#028090" for v in g["diff"]])
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="CH-11 Weighted minus simple average USD/KRW", xlabel="Month", ylabel="KRW", xticks=list(g.index))
plt.show()

## 3-7 환율과 해외 주문 수 · CH-12

In [ ]:
import numpy as np
m = pd.DataFrame({
    "usdkrw": fxd.groupby(fxd.index.month)["USDKRW=X"].mean(),
    "us_orders": t2[t2["country"] == "US"].groupby(t2["date"].dt.month).size(),
})
print(m.round(1))
slope, intercept = np.polyfit(m["usdkrw"], m["us_orders"], 1)
print("slope:", round(slope, 3), "/ corr:", round(m["usdkrw"].corr(m["us_orders"]), 2))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.scatter(m["usdkrw"], m["us_orders"], color="#1F3A5F")
for mon, r in m.iterrows():
    ax.annotate(str(mon), (r["usdkrw"], r["us_orders"]), xytext=(4, 3), textcoords="offset points", fontsize=8)
xs = np.linspace(m["usdkrw"].min(), m["usdkrw"].max(), 50)
ax.plot(xs, slope * xs + intercept, color="#D98E22", label="trend")
ax.set(title="CH-12 USD/KRW vs US orders (monthly)", xlabel="KRW per USD (monthly mean)", ylabel="US orders")
ax.legend()
plt.show()

## 3-8 $0.50 단위로 반올림한 표시가 · CH-13

In [ ]:
usd_rate = rate["USDKRW=X"]
tag["usd_tag"] = (tag["usd"] * 2).round() / 2          # 0.5 단위로 반올림
tag["diff_krw"] = (tag["usd_tag"] * usd_rate - tag["krw"]).round(0)
print(tag[["item", "krw", "usd", "usd_tag", "diff_krw"]])
print("total diff (one each):", tag["diff_krw"].sum())

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3))
ax.bar(tag["item"], tag["diff_krw"], color=["#028090" if v >= 0 else "#C9382A" for v in tag["diff_krw"]])
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="CH-13 Rounded USD tag vs exact price", ylabel="KRW (tag - exact)")
plt.show()

## 3-9 AI 가격표 JSON 검증하고 저장

In [ ]:
import json
with open("ai_pricetag.json", encoding="utf-8") as f:
    ai = json.load(f)
print(ai["base_date"], ai["rates_used"], ai["note"])
a = pd.DataFrame(ai["items"])
r = ai["rates_used"]
# 되돌려 곱해 원화와 비교
# 달러는 0.5달러 단위로 반올림했으므로 0.25달러(반올림 폭)까지, 엔 · 위안은 1% 까지 통과
a["usd_ok"] = (a["usd"] * r["USD"] - a["krw"]).abs() <= 0.25 * r["USD"]
a["jpy_ok"] = (a["jpy"] * r["JPY_per_100"] / 100 / a["krw"] - 1).abs() < 0.01
a["cny_ok"] = (a["cny"] * r["CNY"] / a["krw"] - 1).abs() < 0.01
print(a)
print(a[["usd_ok", "jpy_ok", "cny_ok"]].sum())

In [ ]:
a["jpy"] = (a["krw"] / (r["JPY_per_100"] / 100)).round(-1)     # 1엔 = 9.433원 으로 나누고 10엔 단위
a["jpy_ok"] = (a["jpy"] * r["JPY_per_100"] / 100 / a["krw"] - 1).abs() < 0.01
print(a[["item", "krw", "usd", "jpy", "cny", "jpy_ok"]])
out = a[["item", "krw", "usd", "jpy", "cny"]]
out.to_csv("price_tag_checked.csv", index=False, encoding="utf-8-sig")
print("saved:", len(out), "rows")